## Graph Databse

#### Building a Q&A app over GraphDB

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

In [4]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

In [9]:
# Dataset Movies
movie_query = '''
LOAD CSV WITH HEADERS FROM
"https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv"
AS row

MERGE (m:Movie {id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.directors, '|')|
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|')|
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|')|
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
'''

In [10]:
graph.query(movie_query)

[]

In [11]:
graph.refresh_schema()
print(graph.schema)

Node properties:
CEO {POB: STRING, name: STRING, YOB: INTEGER}
Company {name: STRING}
Entrepreneur {POB: STRING, name: STRING, YOB: INTEGER}
Country {name: STRING}
Person {name: STRING, born: INTEGER}
Movie {title: STRING, released: INTEGER, id: STRING, imdbRating: FLOAT}
User {name: STRING, city: STRING, userId: INTEGER, age: INTEGER}
Post {postId: INTEGER, content: STRING, timestamp: DATE_TIME}
Genre {name: STRING}
Relationship properties:
FRIEND {type: STRING}
LIKES {type: STRING}
The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:POSTED]->(:Post)
(:User)-[:LIKES]->(:User)
(:User)-[:FRIEND]->(:User)


In [12]:
extra = '''
LOAD CSV WITH HEADERS FROM
"https://raw.githubusercontent.com/tomasonjo/blog-datasets/refs/heads/main/movies/movies_small.csv"
AS row
MATCH (m:Movie {id:row.movieId})
FOREACH (director in split(row.director, '|')|
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
'''
graph.query(extra)


[]

In [18]:
graph.refresh_schema()


In [13]:
groq_api_key = os.getenv('GROQ_API_KEY')

In [14]:
from langchain_groq import ChatGroq
llm = ChatGroq(api_key=groq_api_key, model="gemma2-9b-it")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x78ff6fa04260>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x78ff6fa05250>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
from langchain_neo4j import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(llm = llm, graph=graph, verbose=True, allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x78ff714288c0>, cypher_generation_chain=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x78ff6fa04260>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x78ff6fa05250>, model_name='gemma2-9b-it', model_kwargs

In [ ]:
response = chain.invoke({'query':"Who was the director of 'Casino'?"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Casino'})<-[:DIRECTED]-(p:Person) RETURN p.name


Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


{'query': "Who was the director of 'Casino'?",
 'result': "Martin Scorsese  was the director of 'Casino'. \n"}

In [21]:
response = chain.invoke({'query':"Who were the actors in the film 'Casino'?"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(p:Person)RETURN p.name
Full Context:
[{'p.name': 'Robert De Niro'}, {'p.name': 'Joe Pesci'}, {'p.name': 'Sharon Stone'}, {'p.name': 'James Woods'}]

> Finished chain.


{'query': "Who were the actors in the film 'Casino'?",
 'result': "Robert De Niro, Joe Pesci, Sharon Stone, and James Woods were in the film 'Casino'.  \n"}

In [22]:
chain.invoke("actors who acted in multiple movies")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie) WITH p, COUNT(m) AS numMovies GROUP BY p HAVING numMovies > 1 RETURN p.name


CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'GROUP': expected 'FOREACH', ',', 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'DELETE', 'DETACH', 'FINISH', 'INSERT', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REMOVE', 'RETURN', 'SET', 'SKIP', 'UNION', 'UNWIND', 'USE', 'WHERE', 'WITH' or <EOF> (line 1, column 71 (offset: 70))
"MATCH (p:Person)-[:ACTED_IN]->(m:Movie) WITH p, COUNT(m) AS numMovies GROUP BY p HAVING numMovies > 1 RETURN p.name"
                                                                       ^}